# MLOps: Productionizing a ML Pipeline with MLflow

This notebook walks through how to take a simple scikit-learn model from a **notebook experiment** to a **reproducible, trackable production pipeline** using [MLflow](https://mlflow.org/).

We use sklearn's **Wine recognition dataset** — 178 samples, 13 chemical measurements, 3 cultivar classes. It is a different dataset from the Iris lab in ML Theory, but the workflow is the same: *how do we ship and manage models in production?*

**Topics covered:**
1. The production gap (why MLOps matters)
2. Experiment tracking (params, metrics, artifacts)
3. Comparing and searching runs
4. Logging and loading models
5. Model Registry (register, stage, promote)
6. MLflow Projects (packaged, reproducible training)
7. Serving predictions from a registered model


## Setup

Import libraries and point MLflow at a local tracking directory inside this lab folder.

In [ ]:
import shutil
from pathlib import Path

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
LAB_DIR = Path(".").resolve()
MLFLOW_DB = LAB_DIR / "mlflow.db"
ARTIFACTS_DIR = LAB_DIR / "mlartifacts"
TRACKING_URI = f"sqlite:///{MLFLOW_DB.as_posix()}"
EXPERIMENT_NAME = "wine-mlops-lab"
REGISTERED_MODEL_NAME = "wine-classifier"

# Fresh start for this lab session (delete if you want to keep prior runs)
for path in (MLFLOW_DB, ARTIFACTS_DIR):
    if path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)
print("Database:", MLFLOW_DB)

---
## 1. The Production Gap

In the ML Theory notebook you trained a model inline:

```python
model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
```

That works for learning, but in production you quickly need answers to questions like:

| Question | Without MLOps | With MLflow |
|----------|---------------|-------------|
| Which hyperparameters produced this model? | Lost in a notebook cell | Logged as **parameters** |
| How accurate was it on the test set? | Printed once, then gone | Logged as **metrics** |
| Where is the serialized model file? | `model.pkl` on someone's laptop | Stored as a versioned **artifact** |
| Which model is live in production? | "Ask Dave" | **Model Registry** stage: `Production` |
| How do we retrain with the same code + env? | Copy-paste cells | **MLflow Project** |

MLflow does not replace your ML code — it **wraps** it with metadata, storage, and lifecycle management.

In [ ]:
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data,
    wine.target,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=wine.target,
)

print(f"Samples: {wine.data.shape[0]}, Features: {wine.data.shape[1]}, Classes: {len(wine.target_names)}")
print(f"Train samples: {len(y_train)}, Test samples: {len(y_test)}")
print("Feature names:", list(wine.feature_names))

---
## 2. Experiment Tracking

An **experiment** groups related runs (e.g. all attempts to classify wine cultivars).
A **run** is one execution of your training code.

Inside `mlflow.start_run()` you log:
- **Parameters** — inputs you chose (`max_depth`, `test_size`, …)
- **Metrics** — outputs you measured (`accuracy`, `f1_macro`, …)
- **Artifacts** — files (plots, model binaries, data snapshots)

Think of a run as a structured lab notebook entry that never gets lost.

In [ ]:
max_depth = 3
min_samples_split = 2

with mlflow.start_run(run_name="baseline-depth-3") as run:
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_macro", f1)

    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 (macro): {f1:.4f}")

### Browse runs in the UI

MLflow ships with a web UI. From a terminal in this folder, run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open [http://127.0.0.1:5000](http://127.0.0.1:5000) and click the `wine-mlops-lab` experiment.

You will see your run's parameters, metrics, and (once we log them) artifacts.

---
## 3. Comparing Runs

Hyperparameter tuning produces many runs. MLflow lets you search and compare them programmatically — no more scrolling through notebook outputs.

In [ ]:
param_grid = [
    {"max_depth": depth, "min_samples_split": split}
    for depth in [2, 3, 5, 10, None]
    for split in [2, 5, 10]
]

for params in param_grid:
    with mlflow.start_run(run_name=f"depth={params['max_depth']}-split={params['min_samples_split']}"):
        model = DecisionTreeClassifier(
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            random_state=RANDOM_STATE,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_macro DESC"],
)

display_cols = [
    "run_id",
    "tags.mlflow.runName",
    "params.max_depth",
    "params.min_samples_split",
    "metrics.accuracy",
    "metrics.f1_macro",
]
runs_df[display_cols].head(10)

The best run by F1 score becomes our candidate for production. We will promote it in the Model Registry below.

---
## 4. Logging Models as Artifacts

Logging a model saves the serialized estimator **inside the run's artifact store**, together with a `MLmodel` metadata file that records the flavor (`sklearn`), signature, and environment.

Later you can reload it with `mlflow.sklearn.load_model(run_id)` — no manual `joblib.dump` paths.

In [ ]:
def parse_max_depth(value):
    if value is None or str(value) in {"", "None", "nan"}:
        return None
    return int(value)


best_run = runs_df.iloc[0]
best_params = {
    "max_depth": parse_max_depth(best_run["params.max_depth"]),
    "min_samples_split": int(best_run["params.min_samples_split"]),
}

with mlflow.start_run(run_name="best-model-with-artifact") as run:
    model = DecisionTreeClassifier(
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mlflow.log_params(best_params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1_macro", f1_score(y_test, y_pred, average="macro"))

    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        input_example=wine.data[:3],
    )

    best_run_id = run.info.run_id
    model_uri = model_info.model_uri

print(f"Best run ID: {best_run_id}")
print(f"Model URI: {model_uri}")

loaded_model = mlflow.sklearn.load_model(model_uri)
print(f"Reloaded model predicts: {loaded_model.predict(wine.data[:3])}")

---
## 5. Model Registry

The **Model Registry** adds a lifecycle on top of logged models:

| Stage | Typical use |
|-------|-------------|
| `None` | Just registered, under review |
| `Staging` | Candidate for production, undergoing final checks |
| `Production` | Live model serving real traffic |
| `Archived` | Retired, kept for audit |

Registering creates a named model (`wine-classifier`) with **versions**. Promoting a version to `Production` tells your serving code which artifact to load.

In [ ]:
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
print(f"Registered model: {registered.name}")
print(f"Version: {registered.version}")

client.transition_model_version_stage(
    name=REGISTERED_MODEL_NAME,
    version=registered.version,
    stage="Staging",
    archive_existing_versions=False,
)
print(f"Version {registered.version} moved to Staging")

client.transition_model_version_stage(
    name=REGISTERED_MODEL_NAME,
    version=registered.version,
    stage="Production",
    archive_existing_versions=True,
)
print(f"Version {registered.version} promoted to Production")

for mv in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'"):
    print(f"  v{mv.version} -> stage={mv.current_stage}, run_id={mv.run_id}")

In [ ]:
production_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/Production")

sample = wine.data[0].reshape(1, -1)
pred_class = production_model.predict(sample)[0]
cultivar = wine.target_names[pred_class]

feature_df = pd.DataFrame(sample, columns=wine.feature_names)
print("Input:")
display(feature_df)
print(f"Production model prediction: {cultivar}")

---
## 6. MLflow Projects

Notebooks are great for exploration, but production training should be **packaged** so anyone can reproduce it with one command.

An MLflow Project is a folder with:
- `MLproject` — entry points and parameters
- `python_env.yaml` (or `conda.yaml`) — dependencies
- `train.py` — the actual training script

Open the `wine_classifier/` folder next to this notebook and inspect those files.

Run the project from the terminal (use `--env-manager local` to reuse your current Python environment):

```bash
cd wine_classifier
MLFLOW_TRACKING_URI=sqlite:///../mlflow.db mlflow run . --env-manager local -P max_depth=5 -P min_samples_split=2
```

MLflow executes `train.py`, logs a new run, and registers a new model version.

In [ ]:
import os
import subprocess
import sys

project_dir = LAB_DIR / "wine_classifier"
env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "mlflow",
        "run",
        str(project_dir),
        "--env-manager",
        "local",
        "-P",
        "max_depth=4",
        "-P",
        "min_samples_split=2",
        "-P",
        f"experiment_name={EXPERIMENT_NAME}",
    ],
    capture_output=True,
    text=True,
    check=True,
    env=env,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

After the project run, check the MLflow UI again — you should see a new run logged by the packaged training script, and a new version in the Model Registry.

---
## 7. Serving Predictions

Production code should never import training notebooks. Instead it loads the model by **registry stage**:

```python
model = mlflow.sklearn.load_model("models:/wine-classifier/Production")
prediction = model.predict(wine.data[0].reshape(1, -1))
```

This is exactly what you would wire into the FastAPI lab (`labs/02 FastApi/`) — replace the hard-coded `fruit_class = 0` with a real model loaded from the registry.

The `wine_classifier/predict.py` script demonstrates a minimal inference entry point.

In [ ]:
import subprocess

predict_script = project_dir / "predict.py"
result = subprocess.run(
    [sys.executable, str(predict_script)],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

### Optional: MLflow model server

For a quick REST API without writing FastAPI code, MLflow can serve a registered model directly:

```bash
mlflow models serve -m models:/wine-classifier/Production -p 5001 --env-manager local
```

Then POST JSON to `http://127.0.0.1:5001/invocations` (13 features — alcohol, malic_acid, …, proline):

```json
{"inputs": [[14.23, 1.71, 2.43, 15.6, 127.0, 2.8, 3.06, 0.28, 2.29, 5.64, 1.04, 3.92, 1065.0]]}
```


---
## End-to-End Pipeline

Here is the full MLOps loop we just walked through:

```
  ┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐
  │  train.py   │────▶│  MLflow Tracking │────▶│  Model Registry │
  │  (Project)  │     │  params/metrics  │     │  Staging → Prod │
  └─────────────┘     └──────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
                                               ┌─────────────────────┐
                                               │  Serving (FastAPI / │
                                               │  mlflow models serve)│
                                               └─────────────────────┘
```

1. **Develop** — explore in this notebook, log experiments
2. **Package** — move training logic into `wine_classifier/train.py`
3. **Track** — every run records what changed and how well it performed
4. **Register** — promote the best version to `Production`
5. **Serve** — load by stage name, not by file path


---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| Experiment tracking | Log params, metrics, and artifacts for every training run |
| Runs | One execution = one auditable record |
| Model logging | `mlflow.sklearn.log_model` bundles model + metadata |
| Model Registry | Named versions with lifecycle stages (`Staging`, `Production`) |
| MLflow Projects | Reproducible training via `mlflow run` |
| Serving | Load `models:/name/Production` — decouple training from inference |

**Next steps:**
- Wire the registered model into the FastAPI lab for a real `/predict` endpoint
- Try `mlflow ui` to compare runs visually
- Add a validation metric threshold in `train.py` that blocks promotion to `Production` if F1 drops below a target
